# TL-Bot - char_classifier Training (Lightning AI only)

**Before the very first run:** open a Studio with a T4 GPU before running this notebook.

**Every session: run all cells top to bottom.**
- Cell 1 — set scripts and epoch count.
- Cell 2 — confirms persistent studio storage.
- Cell 3 — clones/pulls the repo and starts or resumes training. Child-process output is piped back into the cell; without that, failures surface as a bare `CalledProcessError`.
- Cell 4 — final results once cell 3 completes: run summary, the metrics of the epoch saved as `best.pt`, and `curves.png`. Refuses to report on an unfinished run.

Checkpoints are saved after every epoch and persist across sessions via Lightning's persistent studio storage.

---
**One-time: zip and upload the dataset**
```powershell
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset
# Upload char-dataset.zip to /teamspace/studios/this_studio/TL-Bot/
```

This notebook is Lightning AI-only. For Colab/Kaggle/Local runs, use `colab_train.ipynb` / `kaggle_train.ipynb` / `local_train.ipynb` instead.

---
## Cell 1 - Configure
Set scripts and epoch count for this run. Edit here only.

In [ ]:
SCRIPTS = ["latin"]
# kana/hangul/cjk are real-context-capable as of the runs-8+ migration
# (see the run-history note below) -- for those, also set
# SCHEDULER = "cosine-warm" and ~30-40 epochs.

# Epochs for this run.
# RESUME defaults to True -- checking progress.json before each session (cell 3
# prints it automatically) is how to decide whether continuing is worthwhile or
# hyperparameters need a change, not a hardcoded per-script flag. best.pt only
# ever updates on a genuine score improvement (see train.py), so it can't
# regress from a bad resume or a bad fresh attempt either way.
#
# Latin config history (2026-08-07/08): run 1 (LR 3e-4, unfreeze 4, GRID_MODE
# "all", 60 epochs) peaked val_acc 0.576 at epoch 9 then plateaued. run 2 (LR
# 1e-4, unfreeze 2) converged markedly slower with no proven benefit -- reverted.
# run 3/4 dropped GRID_MODE to "single" to remove grid-rotation augmentation
# variants (rotation-ambiguous letters like b/q, d/p, n/u, 6/9, M/W turned into
# each other under a fixed label) -- confirmed present but not dominant; run 4
# additionally widened TileGrid3x3's crop scale (0.20-0.45 -> 0.40-0.75) to fix
# a "stroke fragment -> low_i" collapse, which only relocated it (0.4814 peak,
# down from run 3's 0.5123). run 5 (char-dataset-ctx-small, GRID_MODE "single")
# pivoted to real string-rendered/target-glyph-cropped tiles instead of
# crop-scale tuning -- 0.7719 @ epoch 24, and a confused-pairs diagnostic
# confirmed the collapse mechanism itself resolved (low_i share of errors
# 42.6% -> 17.5%, remaining errors are ordinary case-ambiguity like I/l, S/s,
# C/c, O/o, V/v, W/w, U/u, Z/z -- not a generic catch-all class). run 5's
# finished checkpoint is preserved at checkpoints/archive/latin/20260820_run1_epoch24/.
#
# run 6 (char-dataset-ctx-small, GRID_MODE "none" -- set in remote_train.py,
# not here) removed the redundant TileGrid3x3 stacking on top of already-real
# context-cropped tiles. CONFIRMED: 0.9023 @ epoch 23 (final), beating run 5
# by +13pts (0.7719 -> 0.9023) and cutting low_i-involved errors further,
# 17.5% -> 5.2% of all errors -- not just a higher baseline, the targeted
# stroke-fragment collapse mechanism itself improved. Residual errors are
# ordinary case/glyph-shape ambiguity (S/s, I/l, 0/O, v/V, w/W). See
# Models/OCR/FINDINGS.md ("Latin Tile-Context Investigation") for the full
# writeup. grid_mode=none + the real-context dataset is now the promoted
# default in train.py/remote_train.py (as plain "char-dataset" -- see that
# module's DATASET_NAME comment for why this notebook still names datasets
# by their original storage-zip names instead).
#
# Current run (run 7): run 6 used the downsampled char-dataset-ctx-small
# (77,465 images, tile_size=64, count-parity with the original char-dataset).
# Run 6's best epoch was 23 of 24 with val_acc still rising
# (epochs_since_best=1 at the final epoch) -- unlike the original 60-epoch
# Latin run, which plateaued with a wasted dead tail, run 6 may have been cut
# off before full convergence. This run switches to the full, uncapped
# char-dataset-ctx and extends the epoch budget to give it room to keep
# climbing. Answers two open questions in one run: does more real-context
# data help further, and was run 6 actually done improving.
#
# Runs 8+ (kana/hangul/cjk real-context migration, 2026-09-01): the same
# isolated-tile -> real-context pivot that fixed Latin (runs 3-7) is being
# applied to the other three scripts. render_chars_context.py already handles
# --scripts kana|hangul|cjk; the blocker was font coverage. Fixed two ways:
# (1) render_chars.py's copy_system_fonts() now includes .ttc/.otc collections
# (MS Gothic, YaHei, YuGothic, MingLiU, MS JhengHei are .ttc-only and were
# silently skipped, starving CJK of fonts), and each .ttc face is expanded
# separately; (2) Noto Sans CJK (28 OTFs, 7 weights x jp/kr/sc/tc, full
# coverage of every kana/top-500 hangul/top-3000 CJK class) added under
# Models/Datasets/google-fonts/, passed via --extra-fonts-dir. For a
# kana/hangul/cjk run: set SCRIPTS below, use SCHEDULER = "cosine-warm"
# (rougher loss surface on few-shot data), ~30-40 epochs. GRID_MODE stays
# unset -> "none" applies, which is correct for real-context data on every
# script (see the GRID_MODE note below). Results will land in FINDINGS.md.
EPOCHS = 36

SCHEDULER = "cosine"   # cosine (recommended) | cosine-warm | none
LR        = 3e-4       # head LR; backbone uses LR * 0.1
RESUME    = True        # resume last.pt; set to False only for a deliberate fresh restart

# GRID_MODE isn't set here -- remote_train.py's own default ("none") is what
# actually applies, since this notebook doesn't pass --grid-mode in cell 3.
# "none" is correct for the real-context dataset below on EVERY script
# (latin and, as of the runs-8+ migration, kana/hangul/cjk too -- real
# string context makes synthetic TileGrid3x3 tiling redundant and, per
# run 6, actively harmful); kept implicit rather than duplicated to match
# remote_train.py's single source of truth for this flag.

# Dataset variant to train against.
#   "char-dataset"            = real-context, ALL scripts (latin+kana+hangul+
#                                cjk) -- the promoted default as of the runs-8+
#                                migration. On Drive, char-dataset.zip was
#                                updated in place to this (a deliberate one-time
#                                break of the frozen-Drive-names rule -- see
#                                remote_train.py's DATASET_NAME comment).
#   "char-dataset-ctx-small"  = run 5/6 Latin-only downsampled real-context
#   "char-dataset-ctx"        = run 7 Latin-only full/uncapped real-context
#   "char-dataset-legacy"     = old isolated-tile pipeline (runs 1-4), local
#                                only now -- no longer on Drive
# Until the refreshed char-dataset.zip is uploaded, a Latin run still needs
# "char-dataset-ctx" explicitly; once it is up, this can drop to the default
# "char-dataset" (all scripts). A non-default name automatically gets its own
# checkpoints/<script>_<suffix> dir (mirrors remote_train.py's _make_ckpt_dir),
# so a comparison run can never overwrite the default dataset's checkpoint.
DATASET_NAME = "char-dataset-ctx"

# Lightning AI: persistent studio storage path.
LIGHTNING_ROOT = "/teamspace/studios/this_studio/TL-Bot"

---
## Cell 2 - Setup
Confirms persistent studio storage.

In [ ]:
import os
os.makedirs(LIGHTNING_ROOT, exist_ok=True)
print(f"Storage ready: {LIGHTNING_ROOT}")

---
## Cell 3 - Train
Clones or pulls the repo, then starts or resumes training.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path as _Path

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"
REPO_DIR = f"{LIGHTNING_ROOT}/Discord-TL_Bot"

# Checkpoint dir, mirroring train.py's scoping. Resolved once here so cell 4 can
# reuse it instead of repeating the rule.
_ALL = {"latin", "kana", "hangul", "cjk"}
_sel = _ALL if "all" in SCRIPTS else set(SCRIPTS)
if _sel >= _ALL:
    CKPT_DIR = _Path(LIGHTNING_ROOT) / "checkpoints"
elif len(SCRIPTS) == 1:
    CKPT_DIR = _Path(LIGHTNING_ROOT) / "checkpoints" / SCRIPTS[0]
else:
    CKPT_DIR = _Path(LIGHTNING_ROOT) / "checkpoints" / "_".join(sorted(_sel))
# Mirrors remote_train.py's _make_ckpt_dir(): a non-default DATASET_NAME gets
# its own checkpoint dir so it can never land in and overwrite a
# default-dataset run's checkpoint.
if DATASET_NAME != "char-dataset":
    _suffix = DATASET_NAME[len("char-dataset"):].lstrip("-_") or DATASET_NAME
    CKPT_DIR = CKPT_DIR.parent / f"{CKPT_DIR.name}_{_suffix}"

os.makedirs(REPO_DIR, exist_ok=True)
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Print last training progress from progress.json before launching -- DO NOT REMOVE
# Wrapped: a display problem must never stop the training launch.
_prog = CKPT_DIR / "progress.json"
try:
    if _prog.exists():
        print("[progress]")
        for k, v in json.loads(_prog.read_text()).items():
            if not isinstance(v, (list, dict)):
                print(f"  {k}: {v}")
    else:
        print("[progress] No prior run found - starting fresh.")
except Exception as e:
    print(f"[progress] Could not read progress.json: {e}")


# Run a child process with its output streamed into the notebook.
#
# subprocess.run() without a pipe is useless here: IPython replaces sys.stdout at
# the Python level only, so a child inherits the kernel's real fd 1 and writes to
# the server log, not this cell. Pipe it and re-print through sys.stdout instead.
def _run_streamed(cmd):
    print("$ " + " ".join(str(c) for c in cmd) + "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, errors="replace")
    tail = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        tail.append(line)
        del tail[:-40]
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f"remote_train.py exited {rc}. Last {len(tail)} lines:\n" + "".join(tail))
    return rc


_cmd = [
    "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
    "--skip-clone",
    "--scripts", *SCRIPTS,
    "--epochs", str(EPOCHS),
    "--scheduler", SCHEDULER,
    "--lr", str(LR),
    "--storage-root", LIGHTNING_ROOT,
    "--dataset-name", DATASET_NAME,
]
if RESUME:
    _cmd.append("--resume")
_run_streamed(_cmd)

---
## Cell 4 - Final Results
Run once cell 3 finishes. Prints the run summary and the last epoch's metrics from `progress.json`, plus `curves.png`.

Says so and stops if the run has not reached its last epoch. Fields are read from the JSON as they come, so metrics added to `train.py` show up without editing this cell.

The test-set report (per-class precision/recall, top-1/3/5, confused pairs) is printed at the end of cell 3 and is not saved to disk.

In [ ]:
import json

# Final results, read from progress.json in CKPT_DIR (resolved in cell 3).
# Keys come from the file, so metrics added to train.py appear without edits here.

_d = json.loads((CKPT_DIR / "progress.json").read_text())
_hist = _d.get("history", [])

if _d.get("completed", 0) < _d.get("total_epochs", 0):
    print(f"[results] Training unfinished: epoch {_d.get('completed')} of "
          f"{_d.get('total_epochs')}. Re-run once cell 3 completes.")
else:
    print("=" * 72)
    print(f" FINAL RESULTS   {CKPT_DIR}")
    print("=" * 72)
    for k, v in _d.items():
        if not isinstance(v, (list, dict)):
            print(f"  {k:<20}: {v}")

    if _hist:
        print(f"\n  --- last epoch ({_hist[-1].get('epoch')}) ---")
        for k, v in _hist[-1].items():
            print(f"  {k:<20}: {v}")

    if (CKPT_DIR / "curves.png").exists():
        from IPython.display import Image, display
        display(Image(filename=str(CKPT_DIR / "curves.png")))